In [ ]:
import numpy as np
from PIL import Image
import pandas as pd
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn as nn
import albumentations as A
from torchvision import transforms
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, FasterRCNNPredictor, AnchorGenerator
import matplotlib.pyplot as plt
import os
from torch.optim import AdamW, LinearLR, CosineAnnealingLR, SequentialLR 
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import matplotlib.patches as patches

First me must specify the paths to all files needed for training and testing of the neural network.


In [ ]:
CSV_PATH_TRAIN = './RIVA/annotations/annotations/train.csv'
CSV_PATH_VAL = './RIVA/annotations/annotations/val.csv'
TRAIN_PATH = './RIVA/images/images/train'
VAL_PATH = './RIVA/images/images/val'
TEST_PATH = './RIVA/images/images/test'

Setting up the CUDA device.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = torch_directml.device()

For the transformations applied initially I decided to use simple ones. The choice of the library Albumentations instead of Pytorch for them is because the latter doesn't support bounding boxes and it would imply implementing the box logic manually.


In [ ]:
transform_train = A.Compose([
    A.Resize(512,512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    
    A.Affine(
            scale=(0.85, 1.15),
            translate_percent={'x': (-0.0625, 0.0625), 'y': (-0.0625, 0.0625)},
            rotate=(-45, 45),
            p=0.5
    ),

    # Color and brightness/contrast transforms

    A.OneOf([
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=1),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
    ], p=0.7),
    
    # Noise transforms
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1),
        A.MotionBlur(blur_limit=5, p=1),
    ], p=0.3),
    
    # Sensor noise transform
    A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.3),

    # Normalization by ImageNet stats (backbone pretrained on ImageNet)

    A.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225], 
        max_pixel_value=255.0, 
        p=1.0
    ),
    
    # Convert to Pytorch tensor
    A.ToTensorV2()
  ], telemetry=False, bbox_params=A.BboxParams(format='pascal_voc', # Specifying the input format
                           label_fields=['labels'], # Specifying the label name
                           min_visibility=0.3     # Box must be 30% visible after crop
                           ))
                           
transform_test = A.Compose([
    A.Resize(512, 512),
    A.Normalize(
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225], 
            max_pixel_value=255.0, 
            p=1.0
        ),
    A.ToTensorV2()
], telemetry=False, bbox_params=A.BboxParams(format='pascal_voc', 
                           label_fields=['labels'] 
                           ))


We will have to build a new Dataset subclass that I chose to name BethesdaDataset because of the name of the classifications.


The required minimum protocol for a Dataset type in Pytorch is __len__ and __getitem__, naturally BethesdaDataset follows it


We first define the dataset class that inherits from PyTorch's `Dataset`. The class needs to be compatible with Faster RCNN, that requires specific output structures `(image, targets_dict)`.


In [ ]:
class BethesdaDataset(Dataset):
    """
    Custom Dataset subclass to handle the Bethesda dataset bounding boxes
    and format them correctly for Faster R-CNN and other detection models.
    """
    def __init__(self, csv_file, root_dir, transforms=None):
        self.root_dir = root_dir
        self.transforms = transforms

        # Load the CSV
        self.df = pd.read_csv(csv_file)

        # Collect unique image IDs
        self.image_ids = self.df['image_filename'].unique()

    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        """
        Retrieves the image and corresponding bounding boxes.
        Returns the image and a target dictionary containing 'boxes' and 'labels'.
        """
        image_id = self.image_ids[idx]

        records = self.df[self.df['image_filename'] == image_id]

        image_path = os.path.join(self.root_dir, image_id) # Example: /images/image_01
        image = np.array(Image.open(image_path).convert('RGB')) # Convert to np array for Albumentations

        H, W, _ = image.shape

        boxes = []
        labels = []

        for _, row in records.iterrows():
            x_center, y_center = row['x'] , row['y']
            width, height = row['width'], row['height']
            class_id = row['class'] + 1

            x_min = x_center - (width / 2)
            y_min = y_center - (height / 2)
            x_max = x_center + (width / 2)
            y_max = y_center + (height / 2)

            x_min = max(0, min(x_min, W))
            y_min = max(0, min(y_min, H))
            x_max = max(0, min(x_max, W))
            y_max = max(0, min(y_max, H))

            if (x_max <= x_min) or (y_max <= y_min):
                continue

            # We have to adapt the center based description to the one used in FasterRCNN
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(class_id) # 0 is reserved for the background

        if self.transforms is not None:
            transformed = self.transforms(image=image, bboxes=boxes, labels=labels)

            image = transformed['image']
            boxes = transformed['bboxes']
            labels = transformed['labels']

        if len(boxes) > 0:
            # If theres boxes, we need to convert them to tensors
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            
            # Safety check: ensure correct shape (N, 4)
            if boxes.ndim == 1 and len(boxes) == 4:
                boxes = boxes.unsqueeze(0) 
                 
            labels = torch.as_tensor(labels, dtype=torch.int64)
            
            # Calculate area and iscrowd
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
            
        else:
            # Case zero boxes
            # We must create an empty tensor with the correct shape (0, 4)
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        tensor_image_id = torch.as_tensor([idx])

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = tensor_image_id
        target["area"] = area
        target["iscrowd"] = iscrowd

        return image, target


By default pytorch stacks images but in this case we can't do that because we can and will have different number of bounding boxes per image.


In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

Setting up the custom datasets.


In [ ]:
train_ds = BethesdaDataset(csv_file=CSV_PATH_TRAIN, root_dir=TRAIN_PATH, transforms=transform_train)
test_ds = BethesdaDataset(csv_file=CSV_PATH_VAL, root_dir=VAL_PATH, transforms=transform_test)

Setting up the loaders.


In [ ]:
train_loader = DataLoader(dataset=train_ds, shuffle=True, batch_size=2, collate_fn=collate_fn)
test_loader = DataLoader(dataset=test_ds, shuffle=False, batch_size=2, collate_fn=collate_fn)

Creating a helper function to get the mean and std of the dataset for normalization.


In [ ]:
def get_mean_and_std(loader):
    """
    Calculates the mean and standard deviation of images across a dataloader.
    """
    channels_sum = torch.zeros(3).float()
    channels_squared_sum = torch.zeros(3).float()
    total_pixels = 0

    for data, _ in tqdm(loader):
        for image in data:
            # Divide by 255 to normalize values between [0,1]
            image = image.double() / 255 

            # Sum over height and width dims
            channels_sum += torch.sum(image, dim=[1, 2]) 
            channels_squared_sum += torch.sum(image**2, dim=[1, 2])
            
            # Number of pixels is Height * Width
            num_pixels = image.shape[1] * image.shape[2] 
            total_pixels += num_pixels

    mean = channels_sum / total_pixels
    variance = (channels_squared_sum / total_pixels) - (mean ** 2)
    std = torch.sqrt(torch.as_tensor(variance))
    
    return mean, std

More helper functions but this time to get visualizations of the ground truth.


In [ ]:
def show_image(image):
    """
    Helper function to visualize a single image tensor.
    """
    plt.figure(figsize=(5.12, 5.12))
    plt.imshow(image.permute(1,2,0))
    plt.show()

def show_image_and_bounding(image, labels):
    """
    Helper function to visualize an image along with its real bounding boxes.
    """
    image_np = image.permute(1, 2, 0).cpu().numpy()
    image_np = image_np.astype(np.uint8)

    image_cv = np.ascontiguousarray(image_np)
    
    image_cv = cv2.cvtColor(image_cv, cv2.COLOR_RGB2BGR)

    boxes = labels[1]["boxes"].tolist()
    for box in boxes:
        start_point = (int(box[0]), int(box[1]))
        print(start_point)
        end_point = (int(box[2]), int(box[3]))
        cv2.rectangle(image_cv, start_point, end_point, color=(0,255,0), thickness=1)

    cv2.imwrite("example_with_bounding_boxes.jpg", image_cv)

    plt.figure(figsize=(5.12, 5.12))
    plt.imshow(cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB))
    plt.show()

Setting up the basic non fine tuned Faster-RCNN with ResNet50 as a backbone.


In [ ]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
    trainable_backbone_layers = 5
)

The following functions configure parameter grouping and optimizers to handle different learning rates for the backbone and head of the model.


In [ ]:
def get_model_params_groups(model):
    """
    Separates model parameters into two groups:
    Group 1: Backbone (ResNet+FPN) 
    Group 2: ROI Heads (RPN + BoxPredictor)
    """
    backbone_params = []
    head_params = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue # Ignore already frozen params
        if "backbone" in name:
            backbone_params.append(param)
        else:
            # RPN and ROI Heads
            head_params.append(param)
            
    return backbone_params, head_params

Adding a function to configure differential learning rates.

In [ ]:
def configure_optimization_differential(model, train_dataloader, epochs):
    """
    Configures the optimizer and schedulers for training the model, applying 
    different learning rates to the pretrained backbone and newly initialized heads.
    """
    # Hyperparameters
    HEAD_LR = 1e-3       # Standard learning rate for new weights
    BACKBONE_LR = 1e-4   # 10x slower for pretrained weights 
    MIN_LR = 1e-6
    WEIGHT_DECAY = 1e-5
    WARMUP_PERCENTAGE = 0.05

    # Get parameter groups
    backbone_params, head_params = get_model_params_groups(model)

    # Define AdamW with parameter groups
    optimizer = AdamW([
        {
            'params': head_params, 
            'lr': HEAD_LR
        },
        {
            'params': backbone_params, 
            'lr': BACKBONE_LR
        }
    ], weight_decay=WEIGHT_DECAY)

    # Calculate Steps
    steps_per_epoch = len(train_dataloader) // accumulation_steps
    total_steps = steps_per_epoch * epochs
    warmup_steps = int(total_steps * WARMUP_PERCENTAGE)
    cosine_steps = total_steps - warmup_steps

    # Schedulers
    # Note: PyTorch schedulers handle LR groups automatically,
    # scaling each group proportionally to its initial LR.
    
    scheduler_warmup = LinearLR(
        optimizer, start_factor=0.001, end_factor=1.0, total_iters=warmup_steps
    )
    
    scheduler_cosine = CosineAnnealingLR(
        optimizer, T_max=cosine_steps, eta_min=MIN_LR
    )

    scheduler = SequentialLR(
        optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[warmup_steps]
    )

    return optimizer, scheduler

Defining a helper function to display images with predicted bounding boxes vs ground truth.

In [ ]:
def visualize_predictions(image, targets, predictions, score_threshold=0.3):
    """
    Visualizes ground truth (green) vs predictions (red) bounding boxes on an image.
    """
    # Convert image to numpy
    img_np = image.cpu().permute(1, 2, 0).numpy()
    
    # Denormalize
    mean = [0.7490, 0.7711, 0.8236]
    std = [0.2226, 0.2171, 0.1551]
    img_np = img_np * std + mean
    img_np = np.clip(img_np, 0, 1)
    
    fig, ax = plt.subplots(1, figsize=(12, 12))
    ax.imshow(img_np)
    
    # Ground truth boxes (GREEN)
    gt_boxes = targets['boxes'].cpu().numpy()
    gt_labels = targets['labels'].cpu().numpy()
    
    for box, label in zip(gt_boxes, gt_labels):
        x1, y1, x2, y2 = box
        width, height = x2 - x1, y2 - y1
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2, edgecolor='green', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1-5, f'GT:{label}', color='green', fontsize=8, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    # Predictions (RED)
    pred_boxes = predictions['boxes'].cpu().numpy()
    pred_labels = predictions['labels'].cpu().numpy()
    pred_scores = predictions['scores'].cpu().numpy()
    
    for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
        if score < score_threshold:
            continue
        x1, y1, x2, y2 = box
        width, height = x2 - x1, y2 - y1
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2, edgecolor='red', facecolor='none', linestyle='--'
        )
        ax.add_patch(rect)
        ax.text(x1, y2+15, f'Pred:{label} ({score:.2f})', color='red', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    plt.title(f'Green=GT | Red=Predictions | GT boxes: {len(gt_boxes)} | Pred boxes: {len(pred_boxes)}')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f'prediction_vs_gt_epoch_{epoch+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

Lastly, setting up the trianing loop with gradient accumulation

In [ ]:
metric = MeanAveragePrecision()

accumulation_steps = 32 # making an effective batch size of 64

num_epochs = 100

optimizer, lr_scheduler = configure_optimization_differential(model, train_loader, num_epochs) 

model.to(device)

for epoch in range(num_epochs):
    
  epoch_loss = 0

  model.train(mode=True)
  optimizer.zero_grad()

  print(f"Epoch {epoch+1}/{num_epochs}")
  for i, (inputs, targets) in enumerate(tqdm(train_loader)):

    inputs = list(image.to(device) for image in inputs) 
    targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

    loss_dict = model(inputs, targets)
    losses = sum(loss for loss in loss_dict.values())


    (losses / accumulation_steps).backward()
    if (i+1) % accumulation_steps == 0: # Implementing gradient accumulation due to lack of compute
      optimizer.step()
      optimizer.zero_grad()
      lr_scheduler.step()

      epoch_loss += losses.item()
      
  avg_loss = epoch_loss / len(train_loader)

  print(f"Average Loss: {avg_loss:.4f}")
  
  model.eval()
  metric.reset()
  
  with torch.no_grad():
      test_img, test_target = next(iter(test_loader))
      test_img = list(img.to(device) for img in test_img)
      
      predictions = model(test_img)
      print(f"\nPrediction Check:")
      print(f"  Num boxes predicted: {len(predictions[0]['boxes'])}")
      print(f"  Scores: {predictions[0]['scores'][:5]}")  # Top 5 scores
      print(f"  Labels: {predictions[0]['labels'][:5]}")  # Top 5 labels
      print(f"  Labels (csv): {predictions[0]['labels'][:5] - 1}")

      #visualize_predictions(test_img[0], test_target[0], predictions[0], score_threshold=0.3)

  with torch.no_grad(): # Every epoch test the model's accuracy with the validation dataset
    for test_inputs, test_targets in tqdm(test_loader, desc='Calculating mAP'):
      test_inputs = list(image.to(device) for image in test_inputs) 
      test_outputs = model(test_inputs)

      test_outputs_cpu = []
      for out in test_outputs:
          # Move each tensor in the dict to CPU
          test_outputs_cpu.append({k: v.cpu() for k, v in out.items()})
          
      test_targets_cpu = []
      for t in test_targets:
          # Original targets could be on GPU or CPU, enforcing CPU
          test_targets_cpu.append({k: v.cpu() for k, v in t.items()})

        # 3. Update metric with data on RAM (CPU)
      #show_image_and_bounding(test_inputs[0], test_targets[0])
      metric.update(test_outputs_cpu, test_targets_cpu)

    result = metric.compute()
    print(f"Validation mAP (IoU=0.50:0.95): {result['map_50']:.4f}")

    if result['map_50'] > 0.8: 
        torch.save(model.state_dict(), f"model_epoch_{epoch+1}.pth")